# Spectral Analysis of Deterministic Signals

*This jupyter notebook is part of a [collection of notebooks](../index.ipynb) on various topics of Digital Signal Processing. Please direct questions and suggestions to [Sascha.Spors@uni-rostock.de](mailto:Sascha.Spors@uni-rostock.de).*

## Summary

* The discrete Fourier transform (DFT) of a complex exponential signal with arbitrary frequency does not only consist of a contribution at this frequency. Except for the case that the frequency is a DFT eigenfrequency, additional contributions are inevitably present throughout the entire spectrum. The maximum of the magnitude spectrum (main lobe) is located approximately at the frequency of the exponential signal. The level of the additional contributions (side lobes) decay typically with increasing distance in frequency from the maximum. This effect is known as **leakage effect** or spectral leakage. The leakage effect is a consequence of considering only a finite number of samples in the DFT.

* Windowing refers to weighting the samples before performing the DFT. The amount of spectral leakage can be controlled by the window function. The rectangular window (equal weighting of all samples) results in the narrowest main lobe of all possible windows. Other window functions show wider main lobes at the benefit of lower or faster decaying side lobes. The choice of a particular window function is, amongst others, a trade-off between width of the main lobe and side lobe level.

* The leakage effect limits the spectral resolution of the DFT. The width of the main lobe of a particular window is linked to its capability to detect two exponential signals with comparable levels and similar frequencies, the decay of the side lobes to detect two exponential signals with disparate levels and dissimilar frequencies. The choice of a window function is a trade-off between high resolution and high dynamic range. It is in general application specific.

* Zero-padding a signal before computing its DFT is equivalent to a bandlimited interpolation in the frequency domain.

* For the short-time Fourier transform (STFT), a signal is split into short segments for which the DFT is subsequently computed. The magnitude of the STFT is known as spectrogram. In spectral analysis, the STFT provides insights into the temporal evolution of the spectral contents of a signal. It is of special interest for signals which change their spectral properties over time (non-stationary signals). The leakage effect applies to each subsequent DFT.

**Interactive Example**

The following interactive example illustrates the impact of the leakage effect on the analysis of signal mixtures. Two exponential signals with different frequencies $\Omega_1$ and $\Omega_2$ and amplitudes $A_1$ and $A_2$ are superimposed

\begin{equation}
x_N[k] = A_1 \cdot e^{\mathrm{j} \Omega_1 k} + A_2 \cdot e^{\mathrm{j} \Omega_2 k}
\end{equation}

where $A_1, A_2 \in \mathbb{R}$. The magnitude of the discrete-time Fourier transform (DTFT) and the DFT are plotted. You can adjust the frequencies and amplitudes of the two signals interactively using the sliders.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interactive_output, FloatSlider, IntSlider, Dropdown
from ipywidgets import HBox, VBox, Label, Layout
from IPython.display import display

windows = {
    "rectangular": np.ones,
    "Hann": np.hanning,
    "Hamming": np.hamming,
    "Blackman": np.blackman,
    "Bartlett": np.bartlett,
}


def plot_leakage_mixture(Om1, L1, Om2, L2, N, window):
    A1 = 10 ** (L1 / 20)  # amplitude of 1st exponential signal
    A2 = 10 ** (L2 / 20)  # amplitude of 2nd exponential signal

    # windowed superposition of two exponential signals
    k = np.arange(N)
    w = windows[window](N)
    x = (A1 * np.exp(1j * Om1 * k) + A2 * np.exp(1j * Om2 * k)) * w

    # DTFT of windowed signal (evaluated by zero-padded FFT)
    Om = np.linspace(0, 2 * np.pi, num=2048, endpoint=False)
    Xw = np.fft.fft(x, 2048)

    # DFT of the windowed signal by FFT
    X = np.fft.fft(x)
    mu = np.arange(N) * 2 * np.pi / N

    # magnitude spectra in dB
    eps = np.nextafter(0, 1)
    Xw_dB = 20 * np.log10(abs(Xw) + eps)
    X_dB = 20 * np.log10(abs(X) + eps)
    top = 20 * np.log10(2 * N) + 5  # fixed axis, maximum for A1 = A2 = 1
    bottom = top - 120

    # plot spectra
    plt.figure(figsize=(10, 8))
    ax1 = plt.gca()

    plt.plot(Om, Xw_dB, label=r"$|X_N(e^{j \Omega})|$")
    plt.stem(
        mu,
        X_dB,
        label=r"$|X_N[\mu]|$",
        basefmt=" ",
        linefmt="C1",
        markerfmt="C1o",
        bottom=bottom,
    )
    plt.ylim([bottom, top])
    plt.legend()

    ax1.set_xlabel(r"$\Omega$")
    ax1.set_ylabel(r"$|X_N|$ in dB")
    ax1.set_xlim([0, 2 * np.pi])
    ax1.grid()

    ax2 = ax1.twiny()
    ax2.set_xlim([0, N])
    ax2.set_xlabel(r"$\mu$", color="C1")
    ax2.tick_params("x", colors="C1")
    plt.show()


# widgets for 1st signal, 2nd signal and window function
controls = dict(
    Om1=FloatSlider(min=0, max=2 * np.pi, step=0.01, value=2.03, description="Ω₁"),
    L1=FloatSlider(min=-80, max=0, step=0.5, value=0, description="A₁ / dB"),
    Om2=FloatSlider(min=0, max=2 * np.pi, step=0.01, value=2.98, description="Ω₂"),
    L2=FloatSlider(min=-80, max=0, step=0.5, value=-20, description="A₂ / dB"),
    N=IntSlider(min=4, max=128, step=1, value=32, description="N"),
    window=Dropdown(options=list(windows), value="rectangular", description="type"),
)


def heading(text):
    return Label(
        text, style=dict(font_weight="bold"), layout=Layout(align_self="center")
    )


ui = HBox(
    [
        VBox([heading("1st signal"), controls["Om1"], controls["L1"]]),
        VBox([heading("2nd signal"), controls["Om2"], controls["L2"]]),
        VBox([heading("window function"), controls["window"], controls["N"]]),
    ]
)
display(ui, interactive_output(plot_leakage_mixture, controls))

Output()

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT). Please attribute the work as follows: *Sascha Spors, Digital Signal Processing - Lecture notes featuring computational examples*.